In [ ]:
# Enhanced Job Application Crew with Additional Specialized Agents
import warnings
warnings.filterwarnings('ignore')

from crewai import Agent, Task, Crew
import os
from dotenv import load_dotenv
from crewai_tools import (
    FileReadTool,
    ScrapeWebsiteTool,
    MDXSearchTool,
    SerperDevTool
)

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = os.getenv('SERPER_API_KEY', '')

# Initialize tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='Resume_Ruchi_Sharma.md')
semantic_search_resume = MDXSearchTool(mdx='Resume_Ruchi_Sharma.md')

# ===== ORIGINAL AGENTS =====
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on job postings to help job applicants",
    tools=[scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "As a Job Researcher, your prowess in navigating and extracting critical "
        "information from job postings is unmatched. Your skills help pinpoint the necessary "
        "qualifications and skills sought by employers, forming the foundation for "
        "effective application tailoring."
    )
)

profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do incredible research on job applicants to help them stand out in the job market",
    tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect and synthesize information "
        "from diverse sources to craft comprehensive personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a resume stand out in the job market.",
    tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they resonate perfectly with the job's requirements."
    )
)

interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="Create interview questions and talking points based on the resume and job requirements",
    tools=[scrape_tool, search_tool, read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Your role is crucial in anticipating the dynamics of interviews. With your ability to formulate key questions "
        "and talking points, you prepare candidates for success, ensuring they can confidently address all aspects of the "
        "job they are applying for."
    )
)

# ===== NEW SPECIALIZED AGENTS =====

# 1. Job Discovery Agent
job_scraper = Agent(
    role="Job Discovery Specialist",
    goal="Find and extract relevant job postings from multiple career websites and job boards",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "You are a master at navigating the complex landscape of job boards and career websites. "
        "Your expertise lies in identifying job postings that match specific criteria across "
        "different platforms, from company career pages to major job boards like LinkedIn, "
        "Indeed, and specialized tech job sites. You understand the nuances of different "
        "job board structures and can adapt your scraping strategies accordingly."
    )
)

# 2. Market Intelligence Agent
market_researcher = Agent(
    role="Tech Market Intelligence Analyst",
    goal="Research market trends, salary ranges, and company insights for informed job applications",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "As a market intelligence specialist, you have deep knowledge of the tech industry "
        "landscape. You track salary trends, company growth patterns, funding rounds, "
        "and industry shifts. Your insights help candidates understand what companies "
        "are really looking for and how to position themselves competitively in the market."
    )
)

# 3. Company Deep Dive Agent
company_researcher = Agent(
    role="Company Culture & Background Researcher",
    goal="Conduct thorough research on target companies to understand their culture, values, and recent developments",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "You specialize in uncovering the essence of companies - their culture, values, "
        "recent news, leadership changes, product launches, and strategic directions. "
        "Your research helps candidates understand not just what a company does, but "
        "how they operate, what they value, and how candidates can align their "
        "applications with company priorities."
    )
)

# 4. Skills Gap Analyzer Agent
skills_analyzer = Agent(
    role="Skills Gap Analysis Expert",
    goal="Identify skill gaps between candidate profiles and job requirements, suggesting improvement strategies",
    tools=[read_resume, semantic_search_resume, search_tool],
    verbose=True,
    backstory=(
        "With a keen analytical mind, you excel at identifying the delta between what "
        "candidates currently offer and what the market demands. You provide actionable "
        "insights on skills development, certification paths, and learning resources "
        "that can bridge these gaps and make candidates more competitive."
    )
)

# 5. Cover Letter Specialist Agent
cover_letter_writer = Agent(
    role="Cover Letter Specialist",
    goal="Create compelling, personalized cover letters that highlight relevant experiences for each job application",
    tools=[read_resume, semantic_search_resume, scrape_tool],
    verbose=True,
    backstory=(
        "You are a master storyteller who can weave together a candidate's experiences, "
        "skills, and aspirations into compelling narratives. Your cover letters don't "
        "just summarize resumes - they tell stories that connect candidates' backgrounds "
        "to specific opportunities, demonstrating clear value propositions and cultural fit."
    )
)

# 6. Application Strategy Agent
application_strategist = Agent(
    role="Application Strategy Coordinator",
    goal="Develop strategic approaches for job applications, including timing, prioritization, and follow-up strategies",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "You understand the strategic aspects of job hunting - when to apply, how to "
        "prioritize opportunities, optimal application timing, and effective follow-up "
        "strategies. Your expertise helps candidates maximize their chances by applying "
        "smart, systematic approaches rather than spray-and-pray methods."
    )
)

# 7. Network Intelligence Agent
network_analyzer = Agent(
    role="Professional Network Analyst",
    goal="Identify networking opportunities and connections that could help with job applications",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "You specialize in mapping professional networks and identifying connection "
        "opportunities. You can find mutual connections, alumni networks, industry "
        "contacts, and other relationship pathways that can provide warm introductions "
        "or insider insights into target companies and roles."
    )
)

# 8. Application Tracker Agent
application_tracker = Agent(
    role="Application Progress Manager",
    goal="Track application statuses, deadlines, and follow-up requirements across multiple job applications",
    tools=[search_tool],
    verbose=True,
    backstory=(
        "You are the organized project manager of job applications. You track deadlines, "
        "application statuses, interview schedules, and follow-up requirements. Your "
        "systematic approach ensures no opportunities slip through the cracks and that "
        "all applications receive appropriate attention and follow-up."
    )
)

# ===== ENHANCED TASKS =====

# Task 1: Job Discovery
job_discovery_task = Task(
    description=(
        "Search and scrape job postings from multiple sources including company career pages, "
        "LinkedIn, Indeed, AngelList, and specialized tech job boards. Focus on finding positions "
        "that match these job titles: {target_job_titles}. For each company in this list: {target_companies}, "
        "also check their career pages directly. Collect up to {max_jobs} relevant positions with "
        "complete job descriptions, requirements, and application details."
    ),
    expected_output=(
        "A comprehensive list of job opportunities with URLs, job titles, company names, "
        "key requirements, application deadlines, and platform source for each position."
    ),
    agent=job_scraper,
    async_execution=True
)

# Task 2: Market Intelligence
market_research_task = Task(
    description=(
        "Research current market conditions for the target job roles: {target_job_titles}. "
        "Analyze salary ranges by experience level and location, identify trending skills "
        "and technologies, research company funding status and growth trajectories, and "
        "gather intelligence on hiring patterns and requirements in the current market."
    ),
    expected_output=(
        "A market intelligence report covering salary benchmarks, trending skills, "
        "company growth patterns, and current hiring market conditions for the target roles."
    ),
    agent=market_researcher,
    async_execution=True
)

# Task 3: Company Deep Dive
company_research_task = Task(
    description=(
        "Conduct in-depth research on each company from the job discovery results. "
        "Research company culture, values, recent news, leadership team, product updates, "
        "funding rounds, and strategic initiatives. Look for information that could be "
        "relevant for tailoring applications and interview preparation."
    ),
    expected_output=(
        "Detailed company profiles including culture insights, recent developments, "
        "leadership information, and strategic priorities for each target company."
    ),
    context=[job_discovery_task],
    agent=company_researcher,
    async_execution=True
)

# Task 4: Enhanced Profile Analysis
enhanced_profile_task = Task(
    description=(
        "Create a comprehensive candidate profile using the GitHub URL ({github_url}), "
        "resume file, and personal writeup ({personal_writeup}). Analyze technical skills, "
        "project experiences, leadership capabilities, and career progression. Cross-reference "
        "with market intelligence to identify competitive advantages and positioning strategies."
    ),
    expected_output=(
        "An enhanced candidate profile with skills inventory, experience highlights, "
        "competitive positioning, and unique value propositions."
    ),
    context=[market_research_task],
    agent=profiler,
    async_execution=True
)

# Task 5: Skills Gap Analysis
skills_gap_task = Task(
    description=(
        "Analyze the gap between the candidate's current skills and the requirements "
        "from the discovered job postings. Identify missing skills, certifications, "
        "or experiences that could strengthen the candidate's applications. Provide "
        "specific recommendations for skill development and learning resources."
    ),
    expected_output=(
        "A skills gap analysis with prioritized recommendations for skill development, "
        "certification paths, and learning resources to improve job application success."
    ),
    context=[job_discovery_task, enhanced_profile_task],
    agent=skills_analyzer
)

# Task 6: Job Requirements Analysis
detailed_job_analysis_task = Task(
    description=(
        "Perform detailed analysis of each job posting from the discovery phase. "
        "Extract and categorize requirements by importance, identify common themes "
        "across similar roles, and create requirement profiles for different job categories."
    ),
    expected_output=(
        "Detailed analysis of job requirements organized by role type, with importance "
        "rankings and common themes identified across similar positions."
    ),
    context=[job_discovery_task],
    agent=researcher
)

# Task 7: Strategic Resume Creation
strategic_resume_task = Task(
    description=(
        "Create multiple targeted resume versions based on the job analysis, candidate profile, "
        "and company research. Develop different resume variants optimized for different job "
        "categories and company types. Ensure each version highlights the most relevant "
        "experiences and skills for specific role types."
    ),
    expected_output=(
        "Multiple tailored resume versions with clear labeling for different job categories, "
        "each optimized to match specific types of positions and company preferences."
    ),
    output_file="strategic_resumes.md",
    context=[detailed_job_analysis_task, enhanced_profile_task, company_research_task],
    agent=resume_strategist
)

# Task 8: Cover Letter Creation
cover_letter_task = Task(
    description=(
        "Create personalized cover letter templates and examples for different types of "
        "positions and companies. Use insights from company research to customize messaging "
        "and demonstrate cultural fit. Create both general templates and specific examples "
        "for high-priority positions."
    ),
    expected_output=(
        "Personalized cover letter templates and specific examples that demonstrate "
        "understanding of company culture and clear value propositions for each role type."
    ),
    output_file="personalized_cover_letters.md",
    context=[strategic_resume_task, company_research_task, detailed_job_analysis_task],
    agent=cover_letter_writer
)

# Task 9: Application Strategy
application_strategy_task = Task(
    description=(
        "Develop a comprehensive application strategy including job prioritization, "
        "application timing, follow-up schedules, and networking approaches. Consider "
        "factors like application deadlines, company hiring cycles, and strategic value "
        "of different opportunities."
    ),
    expected_output=(
        "A strategic application plan with prioritized job targets, optimal timing "
        "recommendations, and systematic follow-up schedules for maximum effectiveness."
    ),
    context=[job_discovery_task, company_research_task, market_research_task],
    agent=application_strategist
)

# Task 10: Network Analysis
network_analysis_task = Task(
    description=(
        "Analyze networking opportunities for each target company. Look for mutual "
        "connections on LinkedIn, alumni networks, GitHub connections, conference "
        "attendees, and other professional relationship pathways that could provide "
        "warm introductions or insider insights."
    ),
    expected_output=(
        "A networking strategy with identified connection opportunities, mutual contacts, "
        "and relationship pathways for each target company and role."
    ),
    context=[company_research_task],
    agent=network_analyzer
)

# Task 11: Comprehensive Interview Preparation
comprehensive_interview_task = Task(
    description=(
        "Create comprehensive interview preparation materials combining insights from "
        "job analysis, company research, and market intelligence. Include technical "
        "questions, behavioral questions, company-specific questions, and strategic "
        "talking points that demonstrate deep understanding of each opportunity."
    ),
    expected_output=(
        "Complete interview preparation guide with technical questions, behavioral "
        "scenarios, company-specific talking points, and strategic positioning advice."
    ),
    output_file="comprehensive_interview_guide.md",
    context=[strategic_resume_task, company_research_task, detailed_job_analysis_task, skills_gap_task],
    agent=interview_preparer
)

# Task 12: Application Tracking Setup
tracking_setup_task = Task(
    description=(
        "Create a comprehensive application tracking system with deadlines, status "
        "updates, follow-up schedules, and progress monitoring for all discovered "
        "opportunities. Include templates for tracking application materials, "
        "interview schedules, and networking contacts."
    ),
    expected_output=(
        "An organized application tracking system with templates and schedules "
        "for managing multiple job applications effectively."
    ),
    output_file="application_tracking_system.md",
    context=[application_strategy_task, job_discovery_task],
    agent=application_tracker
)

# ===== COMPREHENSIVE CREW =====
comprehensive_job_crew = Crew(
    agents=[
        # Core agents
        job_scraper,
        market_researcher,
        company_researcher,
        profiler,
        researcher,
        
        # Specialized agents
        skills_analyzer,
        resume_strategist,
        cover_letter_writer,
        interview_preparer,
        
        # Strategic agents
        application_strategist,
        network_analyzer,
        application_tracker
    ],
    tasks=[
        # Phase 1: Discovery and Research
        job_discovery_task,
        market_research_task,
        company_research_task,
        enhanced_profile_task,
        
        # Phase 2: Analysis
        skills_gap_task,
        detailed_job_analysis_task,
        
        # Phase 3: Content Creation
        strategic_resume_task,
        cover_letter_task,
        
        # Phase 4: Strategy and Preparation
        application_strategy_task,
        network_analysis_task,
        comprehensive_interview_task,
        tracking_setup_task
    ],
    verbose=True
)

# ===== EXECUTION FUNCTION =====
def run_comprehensive_job_hunt(target_job_titles, target_companies, github_url, personal_writeup, max_jobs=20):
    """
    Run the comprehensive job hunting crew with all specialized agents
    
    Args:
        target_job_titles: List of job titles to search for
        target_companies: List of companies to target specifically
        github_url: Candidate's GitHub profile URL
        personal_writeup: Personal description of the candidate
        max_jobs: Maximum number of jobs to analyze
    
    Returns:
        Complete job hunting package with all deliverables
    """
    
    job_hunt_inputs = {
        'target_job_titles': target_job_titles,
        'target_companies': target_companies,
        'github_url': github_url,
        'personal_writeup': personal_writeup,
        'max_jobs': max_jobs
    }
    
    print(f"🚀 Starting comprehensive job hunt for {len(target_job_titles)} role types across {len(target_companies)} companies")
    print(f"📊 Analyzing up to {max_jobs} job opportunities")
    
    # Execute the crew
    result = comprehensive_job_crew.kickoff(inputs=job_hunt_inputs)
    
    print("✅ Comprehensive job hunt analysis completed!")
    print("""
📄 Generated files:
- strategic_resumes.md (Multiple tailored resume versions)
- personalized_cover_letters.md (Cover letter templates and examples)
- comprehensive_interview_guide.md (Complete interview preparation)
- application_tracking_system.md (Organization and tracking tools)
    """)
    
    return result

# ===== EXAMPLE USAGE =====
if __name__ == "__main__":
    # Configuration
    job_titles = [
        "Machine Learning Engineer",
        "Data Scientist",
        "Software Engineer - Machine Learning",
        "AI Engineer",
        "Senior Data Scientist",
        "Principal ML Engineer",
        "Staff Machine Learning Engineer"
    ]
    
    companies = [
        "OpenAI",
        "Anthropic", 
        "Google",
        "Meta",
        "Microsoft",
        "Amazon",
        "Netflix",
        "Uber",
        "Airbnb",
        "Stripe"
    ]
    
    candidate_github = "https://github.com/ruchi1109"
    
    candidate_description = """Experienced Data Scientist/ML Engineer with 3+ years 
    building production-grade machine learning solutions, specializing in LLMs, GenAI, and data automation.
    Proven track record of delivering business-critical ML models that reduce processing time by 95% while
    maintaining high quality standards. Strong focus on practical, iterative development with expertise in 
    Python, R, cloud environments, and model deployment pipelines.
    
    """
    
    # Run comprehensive analysis
    result = run_comprehensive_job_hunt(
        target_job_titles=job_titles,
        target_companies=companies,
        github_url=candidate_github,
        personal_writeup=candidate_description,
        max_jobs=25
    )

🚀 Starting comprehensive job hunt for 7 role types across 10 companies
📊 Analyzing up to 25 job opportunities
 [DEBUG]: == Working Agent: Job Discovery Specialist
 [INFO]: == Starting Task: Search and scrape job postings from multiple sources including company career pages, LinkedIn, Indeed, AngelList, and specialized tech job boards. Focus on finding positions that match these job titles: ['Machine Learning Engineer', 'Data Scientist', 'Software Engineer - Machine Learning', 'AI Engineer', 'Senior Data Scientist', 'Principal ML Engineer', 'Staff Machine Learning Engineer']. For each company in this list: ['OpenAI', 'Anthropic', 'Google', 'Meta', 'Microsoft', 'Amazon', 'Netflix', 'Uber', 'Airbnb', 'Stripe'], also check their career pages directly. Collect up to 25 relevant positions with complete job descriptions, requirements, and application details.
 [DEBUG]: == [Job Discovery Specialist] Task output: 


 [DEBUG]: == Working Agent: Tech Market Intelligence Analyst
 [INFO]: == Starti